# Chapter 10 — Rain Attenuation — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 19. Rain & Fog Attenuation — Ch 10  *(future, outdoor mmWave)*

Skip for indoor; the dominant availability limiter for **outdoor links above ~10 GHz**. Rain fade is
`γ·d` scaled by a nonlinear distance factor. Verified below against Ex 10.1 (ITU) & 10.4 (fog).

- **Specific attenuation** `γ = k·RR^α` dB/km (§10.3.1); k, α are frequency + polarization dependent
  (Table 10.1). Polarization combine (eqs 10.3-10.4): τ = 0° H, 45° circular, 90° V. Horizontal rains
  worse than vertical (elongated drops). → `rain_coeffs()`, `rain_coeff_interp()`, `rain_specific_attenuation()`.
- **ITU model** (eqs 10.5-10.7): `A_0.01 = γ·d·r`, distance factor `r = 1/(1+d/d0)`, effective length
  `d0 = 35·e^(−0.015 RR)`. Availability scaling (eqs 10.8/10.9, by latitude). → `itu_rain_attenuation_db()`,
  `itu_availability_adjust()`. *Verified Ex 10.1: Florida (region N, 95 mm/h), 38.6 GHz, 1.1 km →
  23.8 dB @0.01%, 34.3 dB @0.001%.* Valid to 40 GHz / 60 km.
- **Fog/cloud** `γc = Kl·M` dB/km (eq 10.15). → `fog_attenuation_db()`. *Verified Ex 10.4: 30 GHz, heavy
  fog → 4.7 dB over 15 km.*
- **Crane global model:** two-segment, valid ≤22.5 km, same k/α but different rain regions. ⚠ Its
  eqs 10.11/10.13 are garbled in this OCR (couldn't reproduce Ex 10.2's y=−0.189/43.9 dB), so only the
  distance breakpoint `d(RR)` (eq 10.12) and z (eq 10.14, verified) are given — use Crane 1996/2003 for the full model.


In [ ]:
# Table 10.1: rain regression coefficients (freq_GHz -> kH, kV, aH, aV); used by ITU & Crane
RAIN_COEFFS = {
    2:(6.5e-4, 5.91e-4, 1.121, 1.075), 6:(1.75e-3, 1.55e-3, 1.308, 1.265),
    8:(4.54e-3, 3.95e-3, 1.327, 1.310), 10:(0.0101, 0.00887, 1.276, 1.264),
    12:(0.0188, 0.0168, 1.217, 1.200), 20:(0.0751, 0.0691, 1.099, 1.065),
    30:(0.187, 0.167, 1.021, 1.000), 40:(0.350, 0.310, 0.939, 0.929),
}
def rain_coeff_interp(f_ghz):
    # Interpolate (kH,kV,aH,aV): k on a log-log scale, alpha linear vs log(f) (ITU rule).
    fs = sorted(RAIN_COEFFS); lf = np.log10(f_ghz); lfs = np.log10(fs)
    kH = 10**np.interp(lf, lfs, [np.log10(RAIN_COEFFS[f][0]) for f in fs])
    kV = 10**np.interp(lf, lfs, [np.log10(RAIN_COEFFS[f][1]) for f in fs])
    aH = np.interp(lf, lfs, [RAIN_COEFFS[f][2] for f in fs])
    aV = np.interp(lf, lfs, [RAIN_COEFFS[f][3] for f in fs])
    return kH, kV, aH, aV

def rain_coeffs(k_H, k_V, a_H, a_V, tau_deg, theta_deg=0.0):
    # Polarization/elevation-combined k, alpha (eqs 10.3-10.4). tau: 0=H, 45=circular, 90=V.
    ct = np.cos(np.radians(theta_deg))**2; c2t = np.cos(np.radians(2*tau_deg))
    k = (k_H + k_V + (k_H - k_V)*ct*c2t)/2
    a = (k_H*a_H + k_V*a_V + (k_H*a_H - k_V*a_V)*ct*c2t)/(2*k)
    return k, a

def rain_specific_attenuation(RR_mm_h, k, a):
    return k*RR_mm_h**a                                    # dB/km

kH, kV, aH, aV = rain_coeff_interp(38.6)                   # Ex 10.1 uses horizontal (tau=0)
kh, ah = rain_coeffs(kH, kV, aH, aV, tau_deg=0)
print(f"38.6 GHz horizontal: k={kh:.3f}, alpha={ah:.2f} (book 0.324, 0.95)")


In [ ]:
ITU_RAIN_RATE_001 = {   # Table 10.2: 0.01% rain rate (mm/h) by ITU region
    "A":8, "B":12, "C":15, "D":19, "E":22, "F":28, "G":30,
    "H":32, "J":35, "K":42, "L":60, "M":63, "N":95, "P":145,
}
def itu_rain_attenuation_db(RR_001, k, a, d_km):
    # ITU 0.01% (99.99%) rain fade, eqs 10.5-10.7.
    gamma = rain_specific_attenuation(RR_001, k, a)
    d0 = 35*np.exp(-0.015*min(RR_001, 100))               # eq 10.7 (RR capped at 100 mm/h)
    r  = 1/(1 + d_km/d0)                                  # eq 10.6 distance factor
    return gamma*d_km*r

def itu_availability_adjust(atten_001, availability_pct, low_latitude=False):
    # Scale the 0.01% fade to another availability (eqs 10.8/10.9). low_latitude: |lat| < 30 deg.
    p = 100 - availability_pct                            # outage percentage
    if low_latitude:
        return atten_001*0.07*p**(-(0.855 + 0.139*np.log10(p)))   # eq 10.9
    return atten_001*0.12*p**(-(0.546 + 0.043*np.log10(p)))       # eq 10.8

# Example 10.1: Florida (region N, 95 mm/h), 38.6 GHz horizontal, 1.1 km
a001 = itu_rain_attenuation_db(ITU_RAIN_RATE_001["N"], kh, ah, 1.1)
a5   = itu_availability_adjust(a001, 99.999, low_latitude=True)
print(f"Ex 10.1: A(0.01%)={a001:.1f} dB (book 23.8), A(0.001%, five-nines)={a5:.1f} dB (book 34.3)")


In [ ]:
def fog_attenuation_db(Kl, M_g_m3, d_km):
    return Kl*M_g_m3*d_km                                 # eq 10.15, gamma_c = Kl*M

def crane_breakpoint_km(RR_mm_h):
    return 3.8 - 0.6*np.log(RR_mm_h)                      # eq 10.12, d(RR)  (Crane, partial)

# Example 10.4: 30 GHz, 10 C, heavy fog (M=0.5), 15 km; Kl(30 GHz,10 C)=0.63 (dB/km)/(g/m3)
print(f"Ex 10.4: fog gamma = {fog_attenuation_db(0.63, 0.5, 1):.3f} dB/km -> "
      f"{fog_attenuation_db(0.63, 0.5, 15):.1f} dB over 15 km (book 4.7)")
# Crane breakpoint + z for Ex 10.2 (RR=176): z verified, y/attenuation not (OCR garbled)
rr = 176
print(f"Crane d(RR={rr}) = {crane_breakpoint_km(rr):.3f} km, z = {0.95*(0.026 - 0.03*np.log(rr)):.5f} "
      f"(book z=-0.12266; full Crane atten not reproduced)")
